# 面试问题：Multi-Token Prediction 怎样训练，为什么可能加速解码？

可以直接复述的回答是：第一，标准 LM 每个位置只监督下一个 token，MTP 增加多个未来位置预测头。第二，各头共享上下文表示，但有独立输出层与损失权重。第三，推理时多头给出候选 token，仍需主模型或规则验证可接受前缀。第四，训练要屏蔽越过序列末尾的标签。第五，收益取决于未来 token 的可预测性和验证接受率。第六，要展示各 horizon 的 loss、accuracy 和实际接受 token 数。下面用五段可读 Python 代码训练一个小模型。

## 真实案例：代码补全服务一次提议未来三个 token

五段脱敏代码 token 序列覆盖条件判断、循环、函数返回和日志。教学模型使用小型 Embedding 与三个手写预测头，任务是从当前位置预测未来 1、2、3 个 token。数据量极小且语法模板化，只用于解释 MTP 机制，不能代表真实代码模型质量。

In [1]:
sequences = [  # 定义五段具有明确语义的代码 token 序列
    ["if", "x", ">", "0", ":", "return", "x"],  # 正数条件返回代码
    ["if", "total", ">", "limit", ":", "return", "total"],  # 阈值判断返回代码
    ["for", "item", "in", "items", ":", "print", "item"],  # 遍历并打印元素
    ["def", "add", "(", "a", ",", "b", ")", ":", "return", "a", "+", "b"],  # 两数相加函数
    ["while", "retry", ">", "0", ":", "retry", "=", "retry", "-", "1"],  # 重试计数循环
]  # 结束五段代码补全样本
special = ["<pad>"]  # 定义标签补齐所需的特殊 token
vocabulary = special + sorted({token for sequence in sequences for token in sequence})  # 从可读代码构建固定词表
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到整数编号的映射
print("代码样本：sample | tokens")  # 输入预览展示真实 token 而非裸整数
for index, sequence in enumerate(sequences, start=1):  # 逐条输出五段代码
    print(f"{index} | {' '.join(sequence)}")  # 保留代码结构供学习者理解未来标签
print(f"词表大小={len(vocabulary)}，MTP horizons=[1, 2, 3]")  # 展示模型输出空间和预测跨度


代码样本：sample | tokens
1 | if x > 0 : return x
2 | if total > limit : return total
3 | for item in items : print item
4 | def add ( a , b ) : return a + b
5 | while retry > 0 : retry = retry - 1
词表大小=27，MTP horizons=[1, 2, 3]


## Baseline / 基线：频率式 next-token 只能一步一步生成

基线统计每个 token 后最常出现的下一个 token。即使每次都正确，生成三个 token 仍需要三轮模型调用；歧义上下文只能选择一个高频分支。

In [2]:
from collections import Counter, defaultdict  # 使用计数器实现透明的一步转移基线
next_counts = defaultdict(Counter)  # 收集每个当前 token 的后继频率
for sequence in sequences:  # 遍历五段代码训练数据
    for current, following in zip(sequence[:-1], sequence[1:]):  # 枚举相邻 token 转移
        next_counts[current][following] += 1  # 累加当前 token 的一步后继计数
def baseline_next(token):  # 返回频率最高的一步预测
    if token not in next_counts:  # 未见上下文无法产生可靠补全
        return "<pad>"  # 返回显式未知标记
    return next_counts[token].most_common(1)[0][0]  # 选择计数最高的后继 token
baseline_examples = [(token, baseline_next(token)) for token in ("if", ":", "return", "retry", "for")]  # 选择五个可读上下文查看预测
print("一步基线：current | predicted_next")  # 输出频率模型的补全行为
for current, prediction in baseline_examples:  # 逐项展示五个上下文
    print(f"{current:6} | {prediction}")  # 让冒号后的多分支歧义可见
print("若要提出未来 3 个 token，基线至少需要 3 次串行调用")  # 明确给出解码调用基线


一步基线：current | predicted_next
if     | x
:      | return
return | x
retry  | >
for    | item
若要提出未来 3 个 token，基线至少需要 3 次串行调用


## 核心实现：共享表示与三个未来预测头

把每个有效位置展开成训练样本，并为越过序列尾部的 horizon 设置 mask。模型没有调用 Transformer，一层 Embedding 加三个 Linear 直接展示多头监督。

In [3]:
import torch  # 使用 PyTorch 自动微分训练三个未来预测头
from torch import nn  # 使用基础模块手写小型 MTP 网络
torch.manual_seed(2502)  # 固定初始化保证训练曲线和输出可复现
contexts = []  # 收集所有可作为上下文的当前 token
targets = [[], [], []]  # 分别收集 horizon 1、2、3 的未来标签
masks = [[], [], []]  # 分别记录每个 horizon 是否仍在序列范围内
for sequence in sequences:  # 遍历五段代码构建多 horizon 监督
    ids = [token_to_id[token] for token in sequence]  # 将可读 token 映射为词表编号
    for position in range(len(ids) - 1):  # 最后一个 token 没有未来标签因此跳过
        contexts.append(ids[position])  # 保存当前上下文 token 编号
        for horizon in range(1, 4):  # 为未来一步到三步分别构造标签
            valid = position + horizon < len(ids)  # 判断当前 horizon 是否越过序列尾部
            targets[horizon - 1].append(ids[position + horizon] if valid else 0)  # 越界位置使用 pad 编号占位
            masks[horizon - 1].append(valid)  # 保存损失屏蔽标志
context_tensor = torch.tensor(contexts)  # 构造完整训练上下文张量
target_tensors = [torch.tensor(values) for values in targets]  # 构造三个 horizon 标签张量
mask_tensors = [torch.tensor(values, dtype=torch.bool) for values in masks]  # 构造三个 horizon 布尔 mask
class TinyMTP(nn.Module):  # 定义共享表示和独立未来预测头
    def __init__(self, vocab_size, width=24):  # 初始化小型代码补全模型
        super().__init__()  # 注册 PyTorch 模块状态
        self.embedding = nn.Embedding(vocab_size, width)  # 学习当前 token 的共享上下文表示
        self.heads = nn.ModuleList([nn.Linear(width, vocab_size) for _ in range(3)])  # 为三个未来位置建立独立分类头
    def forward(self, token_ids):  # 计算当前 token 对未来三个位置的 logits
        hidden = torch.tanh(self.embedding(token_ids))  # 用非线性共享表示承载上下文特征
        return [head(hidden) for head in self.heads]  # 返回三个 horizon 的词表分布
model = TinyMTP(len(vocabulary))  # 创建可训练的三头 MTP 模型
loss_curve = []  # 保存训练过程中总损失变化
learning_rate = 0.35  # 选择适合小型全批数据的教学步长
for step in range(121):  # 对确定性小数据执行一百二十次全批更新
    logits_list = model(context_tensor)  # 计算三个未来位置的预测分布
    losses = []  # 收集每个 horizon 的有效位置损失
    for logits, labels, mask in zip(logits_list, target_tensors, mask_tensors):  # 同步处理三组标签和 mask
        losses.append(nn.functional.cross_entropy(logits[mask], labels[mask]))  # 只对未越界位置计算交叉熵
    loss = sum(losses) / len(losses)  # 使用等权平均得到 MTP 总目标
    loss_curve.append(float(loss.detach()))  # 保存当前总损失供曲线观察
    if step < 120:  # 最后一步只评估而不继续更新
        loss.backward()  # 计算共享表示和三个头的梯度
        with torch.no_grad():  # 手工 SGD 更新不记录新的计算图
            for parameter in model.parameters():  # 遍历所有可训练权重
                parameter -= learning_rate * parameter.grad  # 沿负梯度方向更新参数
                parameter.grad = None  # 清空梯度避免跨步累积
print("训练损失：step | total_loss")  # 输出可观察的优化过程
for step in (0, 20, 40, 80, 120):  # 选择五个关键检查点
    print(f"{step:3} | {loss_curve[step]:.4f}")  # 展示多 horizon 目标是否稳定下降


训练损失：step | total_loss
  0 | 3.4383
 20 | 2.6001
 40 | 2.0205
 80 | 1.3147
120 | 0.9411


## 失败案例与修正：歧义上下文不能盲收三个候选

冒号后可能是 return、print 或赋值，远期头置信度会下降。MTP 只能提出候选，验证器按主头概率逐 token 接受；低于阈值立即停止并回到标准解码。

In [4]:
def propose(token):  # 使用三个预测头为当前 token 提出未来候选
    token_id = torch.tensor([token_to_id[token]])  # 把可读上下文映射为单样本张量
    with torch.no_grad():  # 推理阶段不建立梯度图
        logits_list = model(token_id)  # 计算未来一步到三步的词表 logits
        probabilities = [torch.softmax(logits, dim=-1)[0] for logits in logits_list]  # 转换为三个归一化概率分布
    rows = []  # 收集每个 horizon 的候选和置信度
    for horizon, probability in enumerate(probabilities, start=1):  # 逐头读取最大概率 token
        confidence, token_index = probability.max(dim=-1)  # 获取当前 horizon 的 Top-1 候选
        rows.append((horizon, vocabulary[int(token_index)], float(confidence)))  # 恢复可读 token 并保存置信度
    return rows  # 返回三个带置信度的未来候选
ambiguous_proposal = propose(":")  # 对训练集中多分支的冒号上下文提出三个 token
confidence_threshold = 0.72  # 设置教学验证器的最低接受置信度
accepted_tokens = []  # 收集连续通过验证的候选前缀
for horizon, token, confidence in ambiguous_proposal:  # 按 horizon 顺序验证候选
    if confidence < confidence_threshold:  # 第一个低置信候选终止后续批量接受
        break  # 回退主模型的一步解码以避免错误扩散
    accepted_tokens.append(token)  # 保存通过置信门禁的候选
print("歧义上下文 ':' 的候选：horizon | token | confidence")  # 展示多头预测的中间结果
for row in ambiguous_proposal:  # 逐头输出候选和置信度
    print(f"{row[0]} | {row[1]:8} | {row[2]:.3f}")  # 观察远期预测的不确定性
print("验证后批量接受：", accepted_tokens, "其余位置回退标准解码")  # 展示失败保护如何限制错误前缀


歧义上下文 ':' 的候选：horizon | token | confidence
1 | return   | 0.549
2 | a        | 0.202
3 | +        | 0.350
验证后批量接受： [] 其余位置回退标准解码


## 结果表：三个 horizon 的准确率与可接受长度

In [5]:
with torch.no_grad():  # 在固定训练集上计算三个头的确定性指标
    final_logits = model(context_tensor)  # 获取所有上下文的最终预测
accuracies = []  # 收集 horizon 1、2、3 的 Top-1 准确率
print("horizon | valid_labels | top1_accuracy")  # 输出多 token 预测难度随距离变化的结果表
for horizon, logits, labels, mask in zip((1, 2, 3), final_logits, target_tensors, mask_tensors):  # 逐头评估有效标签
    predictions = logits.argmax(dim=-1)  # 获取当前 horizon 的 Top-1 token
    accuracy = float((predictions[mask] == labels[mask]).float().mean())  # 计算屏蔽尾部后的准确率
    accuracies.append(accuracy)  # 保存分项准确率供回归测试
    print(f"{horizon} | {int(mask.sum()):3} | {accuracy:.1%}")  # 展示远期标签数量和准确率
proposal_rows = {token: propose(token) for token in ("if", "return", "for", "retry", ":")}  # 对五个常见上下文生成候选
mean_confidence = sum(row[2] for rows in proposal_rows.values() for row in rows) / 15  # 汇总十五个候选的平均置信度
print(f"五个上下文、十五个候选平均置信度：{mean_confidence:.3f}")  # 输出解码侧可用于阈值校准的指标


horizon | valid_labels | top1_accuracy
1 |  38 | 76.3%
2 |  33 | 78.8%
3 |  28 | 82.1%
五个上下文、十五个候选平均置信度：0.388


## 结果解读

三个头共享当前 token 表示，却分别学习未来 1、2、3 步目标；总损失稳定下降。远期准确率通常更低，因为单 token 上下文不足以确定完整语法分支。冒号案例展示了为什么不能直接提交三个候选：接受长度必须由验证器决定，MTP 的吞吐收益来自高接受率场景。

## 生产边界

真实 MTP 会基于 Transformer 隐藏状态、位置编码和更复杂的头间连接训练，并与主模型验证或 speculative decoding kernel 集成。需要评估训练额外 FLOPs、词表投影显存、接受长度分布和批处理不变性。本例数据极小且只看当前 token，不用于衡量真实代码补全效果。

## 最小回归测试

In [6]:
assert len(sequences) >= 5  # 保证训练案例包含至少五段可读代码
assert len(model.heads) == 3  # 保证模型确实包含三个未来预测头
assert loss_curve[-1] < loss_curve[0] * 0.5  # 保证多 horizon 训练目标明显下降
assert all(0.0 <= accuracy <= 1.0 for accuracy in accuracies)  # 保证三个头的准确率均为有效概率指标
assert len(accepted_tokens) <= 3  # 保证验证器不会接受超过 MTP 提议长度的 token
assert len(target_tensors[2]) == len(context_tensor)  # 保证第三 horizon 与上下文样本一一对应并由 mask 控制尾部
